In [ ]:
using CSV, DataFrames, QuadGK, BenchmarkTools, NLsolve
using Plots, LaTeXStrings; 
using PyCall
using JuMinuit
using Distributions, LinearAlgebra, Distances, SpecialFunctions

# set default values for the pyplot backend of Plots
pyplot(frame=:box, minorticks=true, size=(500,350), titlefont = (11,"serif"), 
    guidefont = (10,"serif"), tickfont = (11,"serif"), 
    legend_font_pointsize=9, legend_font_family="serif",
    bg_color_legend = RGBA(1,1,1,0.15), markerstrokecolor= :auto) 
# mathtext.fontset; supported: ['dejavusans', 'dejavuserif', 'cm', 'stix', 'stixsans', 'custom']
const plt = Plots.PyPlot
plt.matplotlib.rc("mathtext", fontset="stixsans")      
plt.matplotlib.rc("font", family="serif", size=11) 

warnings = pyimport("warnings"); warnings.filterwarnings("ignore") # this will suppress all warning messages

In [ ]:
include("hadronmasses.jl")
const mρ = 0.750unit_choice; const mω = 0.782unit_choice;

In [ ]:
"""
    quadgauss(f, x::T, w::T) where {T<:Vector{Float64}}

Integration of function `f` using the Gaussian quadratures `x` with weights `w`.
`x` and `w` can be generated using, e.g., `gauss(N, a, b)` in the package `QuadGK`.
Using `quadgk` directly from that package causes memory allocation.
However, if the integration region `[a, b]` is fixed, this function does not lead to any allocation and thus is much faster.
"""
function quadgauss(f, x::T, w::T) where {T<:Vector{Float64}}
    res = zero(f(x[1]))  # zero of the same type as f(x[1]), to avoid type instability
    for i in eachindex(x)
        res += f(x[i]) * w[i]
    end
    return res
end

# in this way, each time when we want to change the number of points to n, we simply write
# XW[] = gauss(n, a, b) before the function containing the integration
const XW = Ref(gauss(64, 10, -10));

In [ ]:
# read the data from file and wrap that into Data defined in IMinuit
data_read=DataFrame(CSV.File("data.csv", header=["w", "y", "err"]))
const data =Data(data_read)

In [ ]:
@plt_data data

# Two-channel fit

We consider a fit with two-channel $T$-matrix: $J/\psi \rho$ and isoscalar $D^0\bar D^{*0}$. 
Then the production amplitude may be written as
\begin{align}
  {\cal A}_a = P_\rho + P_\rho G_\rho T_{11}.
\end{align}

We may consider that the background term and the driving amplitude to be different since the $J/\psi\omega$ can also contribute to the production of $D\bar D^*$, and thus have
\begin{align}
  {\cal A}_b = P_0 + P_1 G_\rho T_{11} \simeq P_o + P_1^r T_{11},
\end{align}
where $\operatorname{Im}G_\rho \propto k$ has been neglected in comparison with the $\operatorname{Re}G_\rho \propto \Lambda$, and $P_1\propto 1/\Lambda$ for a multiplicative renormalization to achieve a $\Lambda$ independence of the production amplitude.
\begin{align}
  T_{11}^{\rm LO}(E) = \frac{-8\pi \Sigma_2 \left(1/a_{22} - i \sqrt{2\mu_2 E}\right)}{ \left(1/a_{11} - ik_1\right) \left(1/a_{22,\rm eff} - i\sqrt{2\mu_2 E} \right) },
\end{align}
where
\begin{align}
\frac{1}{a_{22,\rm eff}} = \frac{1}{a_{22}} - \frac{a_{11}}{a_{12}^2\left(1+a_{11}^2 k_1^2\right)} - i \frac{a_{11}^2 k_1}{a_{12}^2\left(1+a_{11}^2 k_1^2\right)}.
\end{align}

$a_{22,\rm eff}$ means the S-wave $D\bar D^*$ scattering length with inelastic effects, and $a_{22}$ means the single-channel $D\bar D^*$ scattering length.

<!-- It may be simply written as
\begin{align}
\frac{1}{a_{22,\rm eff}} = \frac{1}{a_{22}} - b - i {a_{11} k_1} b,
\end{align}
so that
\begin{align}
 b = \left(\frac{1}{a_{22}} - \frac{1}{a_{22,\rm eff}}\right)/(1+i a_{11} k_1)
\end{align}
 -->
<!-- Then the amplitude for fit may be written as
\begin{align}
  {\cal A}^{\rm LO}(E) = P_0 + P_1' \frac{ \left(1/a_{22} - i \sqrt{2\mu_2 E}\right)}{ \left(1/a_{11} - ik_1\right) \left(1/a_{22,\rm eff} - i\sqrt{2\mu_2 E} \right) },
\end{align} -->

**Prediction** of this scenario:

If the $X(3872)$ is detected in the open-charm final state, $e^+e^- \to X(3872) \to D^0 \bar D^0\pi^0$, there should be a threshold enhancement, instead of a dip. 

In [ ]:
# redefine sqrt so that its cut is along the positive x axis
# function xsqrt(x)
#     imag(x) >=0 ? sqrt(x+0im) : -sqrt(x-0im)
# end
function xsqrt(x)
    imag(x) >=0 ? sqrt(x+0im) : -sqrt(x-0im)
end

λ(x,y,z) = x^2 + y^2 + z^2 - 2x*y - 2y*z - 2z*x
qsq(E,m1,m2) = λ(E^2,m1^2,m2^2)/(4E^2)

In [ ]:
function a22eff(e, a11, a12, a22; mrho = mρ)
    k1 = xsqrt(qsq(e, mjψ, mrho)) 
    inva22eff = 1/a22 - a11/a12^2/(1+a11^2*k1^2) - 1im * a11^2*k1/a12^2/(1+a11^2*k1^2)
    return 1/inva22eff
end

function t11_lo(e, a11, a12, a22; mrho = mρ)
    Σ2 = md0 + mdstar0
    μ2 = md0 * mdstar0 / Σ2
    k1 = xsqrt(qsq(e, mjψ, mrho)) 
    k2 = xsqrt(2μ2*(e-md0-mdstar0))
    _inva22eff = 1/a22eff(e, a11, a12, a22)
    return -8π * Σ2 * (1/a22-1im*k2) / (1/a11-1im*k1) / (_inva22eff - 1im*k2)
end

In [ ]:
resolution(x, x0, σ) = exp(-0.5(x-x0)^2/σ^2) / (sqrt(2π)*σ)
resolution(x, σ) = exp(-0.5(x)^2/σ^2) / (sqrt(2π)*σ)

## Fits w/ an energy resolution

with a22eff fixed to a22eff = a_DD* = (-6.39 + 11.74 i) fm

In [ ]:
# fix a22eff = a_DD* = (-6.39 + 11.74 i) fm
function t11_lo_constained(e, a11, a22, a22eff; mrho = mρ)
    Σ2 = md0 + mdstar0
    μ2 = md0 * mdstar0 / Σ2
    k1 = xsqrt(qsq(e, mjψ, mrho)) 
    k2 = xsqrt(2μ2*(e-md0-mdstar0))
    return -8π * Σ2 * (1/a22-1im*k2) / (1/a11-1im*k1) / (1/a22eff - 1im*k2)
end


Assuming that $a_{11,12,22}$ are all real parameters, the value of a22eff (real and imaginary parts) fixes two conditions, and only one free real parameter is left. We choose it to be $a_{22}$.
The relation is
\begin{align}
\frac1{a_{11}} = \frac{k_1}{\operatorname{Im}(1/a_{22,\text{eff}})} \left(\operatorname{Re}\frac1{a_{22,\text{eff}}} - \frac1{a_{22}} \right).
\end{align}

We can also express $a_{12}$ in terms of other parameters, constrained by $a_{22,\text{eff}}$
\begin{align}
  a_{12}^{-2} = \left(\frac1{a_{22}} - \frac1{a_{22,\text{eff}}}  \right)\left(\frac1{a_{11}} - ik_1  \right).
\end{align}

In [ ]:
# here we define r = p1r/p0, p0 is an overall normalization constant
# introduce a non-interfering bg   Nov.08, 2024
function model1(x, par; mrho = mρ)
    p0, r, a22 = par
    a22 = a22/ħc
    a22eff = (-6.39 + 11.74im) /ħc; inv_a22eff = 1/a22eff
    k1 = real( xsqrt(qsq(x, mjψ, mrho)) ) 
    a11 = 1/( k1/imag(inv_a22eff) * (real(inv_a22eff) - 1/a22) ) 
    amp2 = p0^2 * abs2(1+ r * t11_lo_constained(x, a11, a22, a22eff; mrho)) 
    return amp2
end

XW[] = gauss(128, 10, -10)
function res_model1(x, σ, par; mrho = mρ)
    _y, _w = XW[]
    return quadgauss(y -> model1(x-y, par; mrho) * resolution(y, σ), _y, _w)
    # return quadgauss(y -> model1(y, par; mrho) * resolution(x-y, σ), _y, _w)
    # return quadgk(y -> model1(x-y, par) * resolution(y, σ), 10, -10, rtol=1e-3)[1]
end

res_model1(x, par; mrho = mρ) = res_model1(x, 1.7, par; mrho)

In [ ]:
res_model1(3865, (-3.9,0.0025,-2.2)), res_model1(3865,  (-3.9,0.0025,-2.2), mrho=440)

In [ ]:
# to use an adaptive integration method to have better precision over possible narrow pole
function res_model1_gk(x, σ, par; mrho=mρ)
    _y, _w = XW[]
    # return quadgauss(y -> model1(x-y, par) * resolution(y, σ), _y, _w)
    return quadgk(y -> model1(x-y, par; mrho) * resolution(y, σ), -10, 10, rtol=1e-3)[1]
end
res_model1_gk(x, par; mrho=mρ) = res_model1_gk(x, 1.7, par; mrho)

In [ ]:
res_model1_gk(3865, (-3.9,0.0025,-2.2)), res_model1_gk(3865, (-3.9,0.00,-2.2))

In [ ]:
# this is the correct choice.
mrho_model4 = 775 - 75im
res_fit1_4 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data,(3., 0.0001, -4); 
    name =[:p0, :r, :a22], error=[0.1, 0.01, 1], fix_a22 = false )
migrad(res_fit1_4)
# hesse(res_fit1_4)
simplex(res_fit1_4)
migrad(res_fit1_4)
minos(res_fit1_4)

In [ ]:
bestfit_pars_fit1_4 = args(res_fit1_4)

In [ ]:
chisq((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data, bestfit_pars_fit1_4)

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(3805:0.2:3900, x-> res_model1_gk(x, args(res_fit1_4); mrho=mrho_model4),  lab = "Best fit")
plot!(3805:0.2:3900, x-> model1(x, args(res_fit1_4); mrho=mrho_model4),  l = (:dash), lab = :none)

xlabel!(L"\sqrt{s}"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow \pi^{+} \pi^{-} {J} / \psi\right)"*" (pb)")
# savefig("fit.pdf")

In [ ]:
# in units of fm
function a11a12(a22, mrho)
    a22 = a22/ħc
    a22eff = (-6.39 + 11.74im) /ħc; inv_a22eff = 1/a22eff
    x = md0 + mdstar0
    k1 = real( xsqrt(qsq(x, mjψ, mrho)) ) 
    inv_a11 = k1/imag(inv_a22eff) * (real(inv_a22eff) - 1/a22)
    a12 = ( (1/a22 - inv_a22eff) * (inv_a11 - 1im*k1) )^(-0.5)
    (abs(imag(a12)) > 1e-6) ? println(a12) : (a12 = real(a12))
    return (1/inv_a11, a12) .* ħc
end

In [ ]:
a11a12(bestfit_pars_fit1_4[2], mrho_model4)

## Error analysis

### Primer

The purpose here is to generate many parameter sets within $1\sigma$ confidence level (68%). All these parameter sets lead to a $\chi^2$ value $\leq \chi^2_\text{best} + \delta\chi^2$. If there is only one fitting parameter, then  $\delta \chi^2=1$. If there are more than one fitting parameters, then the $\delta \chi^2$ value is different. 
The following table is copied from p.7 of [this lecture notes](http://star-www.st-and.ac.uk/~kdh1/ada/ada08.pdf)

| Confidence level | Probability | # of parameters=1 | 2 | 3 | 4 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| $1\sigma$ | $68 \%$ | 1 | 2.30 | 3.53 | 4.72 |
| $2\sigma$ | $95.4 \%$ | 4 | 6.17 | 8.02 | 9.70 |
| $3\sigma$ | $99.73 \%$ | 9 | 11.8 | 14.2 | 16.3 |

These values can be computed, see below.

In [ ]:
@doc raw"""
    conf_lev(rsq, k::Int=1) 

Confidence level for a given squared Mahalanobis distance `rsq` (``\delta\chi^2``) and dimensionality `k`.

`rsq` is defined as
```math
    (\boldsymbol{x}-\boldsymbol{\mu})^T \Sigma^{-1} (\boldsymbol{x}-\boldsymbol{\mu}),
```
with ``\Sigma`` the covariance matrix, ``\boldsymbol{x}`` a `k`-dimensional vector 
and ``\boldsymbol{\mu}`` the mean vector.
"""
function conf_lev(rsq, k::Int=1) 
    if k ≤ 0
        throw(DomainError(k, "k must be a positive integer."))
    elseif k == 1
        erf(sqrt(rsq/2)) 
    elseif k == 2
        1 - exp(-rsq/2)
    else
        conf_lev(rsq, k-2) - (sqrt(rsq/2))^(k-2) * exp(-rsq/2)/gamma(k/2)
    end
end

# can also be defined with parametric type
# function conf_lev(rsq, ::Type{Val{N}}) where N
#     d::Int = N
#     return conf_lev(rsq, Val{d-2}) - (sqrt(rsq/2))^(d-2) * exp(-rsq/2)/gamma(d/2)
# end
# conf_lev(rsq, ::Type{Val{1}})= erf(sqrt(rsq/2)) 
# conf_lev(rsq, ::Type{Val{2}}) = 1 - exp(-rsq/2) 
# conf_lev(rsq, d::Int=1) = conf_lev6(rsq, Val{d})

In [ ]:
# then we can compute the confidence level for multivariate normal distribution of dimensionality d
let n = 8
    DataFrame(dim = 1:n, oneσ = map(x->conf_lev(1, x), 1:n), twoσ = map(x->conf_lev(4, x), 1:n),
     threeσ = map(x->conf_lev(9, x), 1:n))
end

In [ ]:
conf_lev(9,1)

In [ ]:
# np: number of parameters; n: how many σ's
function δchisq(n, np)
    _prob = conf_lev(n^2, 1); _initval = 3.0
    sol = nlsolve(x-> conf_lev(x[1],np) - _prob, [_initval])
    sol.zero[1]
end

In [ ]:
# the χ^2 value that needs to be increased for error estimates
let n = 30
    DataFrame(Npara = 1:n, oneσ = map(x->δchisq(1, x), 1:n), twoσ = map(x->δchisq(2, x), 1:n),
     threeσ = map(x->δchisq(3, x), 1:n))
end

### Fit 1

In [ ]:
res_fit1_4_1 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data,(3., 0.0001, -5); 
    name =[:p0, :r, :a22], error=[0.1, 0.01, 1], fix_a22 = false )
migrad(res_fit1_4_1)
minos(res_fit1_4_1)

In [ ]:
[res_fit1_4.values...], res_fit1_4.fval

In [ ]:
# Monte Carlo sampling parameter sets; use Symmetric to avoid a little numerical asymmetry in the covariance matrix returned by minuit
function sampled_psets(fit, n)
    cov = matrix(fit)
    return rand(MvNormal(args(fit), Symmetric(cov)), n)'
end

# the Mahalanobis distance 
function mdistance(fit, psets)
    mean_val = [fit.values...]
    cov = matrix(fit)
    [mahalanobis(x, mean_val, inv(Symmetric(cov))) for x in eachrow(psets)];
end

mdistance(mean_val, cov, psets) = [mahalanobis(x, mean_val, inv(Symmetric(cov))) for x in eachrow(psets)];

# parameter sets within nσ
psets_mdgood(psets, mdis, n) = psets[findall(<=(n), mdis), :]

In [ ]:
# Monte Carlo sampling 30000 parameter sets
@time psets = sampled_psets(res_fit1_4_1, 30000);

In [ ]:
# so MINUIT did not give good estimates of the errors
function psets_plt(psets)
    pplt = Matrix{Any}(undef, 3, 3)
    for i = 1:3, j = 1:3
        pplt[i,j] = histogram2d(psets[:,i], psets[:,j], xlab="p$i", ylab="p$j", label=false, bins=100)
    end
    plot(pplt[1,2], pplt[1,3], pplt[2,3],  cb=false, 
    size=(800,600), layout = (3) )
end

In [ ]:
@time psets_plt(psets)

In [ ]:
# we define δχ² ≡ χ² - χbest²
δχ2 = [chisq((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data, x) - res_fit1_4.fval for x in eachrow(psets) ]
# choose the parameter sets resulting in χ²≤ χbest²+3.53 (calculated above in the Primer subsection)
psets_1σfinal3p = psets[findall(<=(3.53), δχ2), :]

In [ ]:
# One sees that the shapes of the parameter contours are different from ellipses, indicating a highly nonlinearality in the dependence of parameters
@time psets_plt(psets_1σfinal3p)

In [ ]:
# the central value, minus and plus errors
function pval_errors(fit, psets)
    p1band = psets[:,1]
    p2band = psets[:,2]
    p3band = psets[:,3]
    a11band = zeros(length(p3band))
    a12band = zeros(length(p3band))
    for i in eachindex(p3band)
        a11band[i], a12band[i] = a11a12(p3band[i], mrho_model4)
    end
    a11, a12 = a11a12(args(fit)[3], mrho_model4)
    p1, p2, p3 = args(fit)
    DataFrame("paras" => ["N [pb]", :R, "a22 [fm]", "a11 [fm]", "a12 [fm]"],
        "central" => [p1, p2, p3, a11, a12],
        "errors" => [extrema(p1band) .- p1,
            extrema(p2band) .- p2,
            extrema(p3band) .- p3,
            extrema(a11band) .- a11,
            extrema(a12band) .-a12])
end

- Now we choose all the parameter sets with the first parameter positive, since the others are located in a disconnected region and should be a different fit (see the above plots)
- In fact, this provides a way to find different fits, and we found two more different fits in this way.

In [ ]:
pval_errors(res_fit1_4, psets_1σfinal3p[findall(psets_1σfinal3p[:,1] .>0), :])

In [ ]:
# for plotting error band
function yerr(x, fit, psets)
    yarr = [res_model1_gk(x, p; mrho=mrho_model4) for p in eachrow(psets)]
    return extrema(yarr) .- res_model1_gk(x, args(fit); mrho=mrho_model4)
end

In [ ]:
# compute error band
xarr = 3805:0.5:3900;
ybands1_3p = [yerr(x, res_fit1_4, psets_1σfinal3p) for x = xarr];
ribbon1_low = [ybands1_3p[i][1] for i in eachindex(ybands1_3p)] 
ribbon1_up = [ybands1_3p[i][2] for i in eachindex(ybands1_3p)];

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(xarr, x-> res_model1_gk(x, args(res_fit1_4); mrho=mrho_model4), lw = 1.5, lab = "Fit 1"; ribbon = (-ribbon1_low, ribbon1_up))
plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_4); mrho=mrho_model4),  l = (:dash), lab = :none)
xlims!(3805, 3900)
xlabel!(raw"$\sqrt{s}$"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow  {J} / \psi \pi^{+} \pi^{-}\right)"*" (pb)")
# savefig("fit1_3p.pdf") 

### Fit 2

In [ ]:
# a different fit, found by a large sampling signaled by some parameter sets far from others
res_fit1_4_2 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data,(0.3, 0.1, -9); 
    name =[:p0, :r, :a22], error=[0.1, 0.01, 1], fix_a22 = false )
migrad(res_fit1_4_2)

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(3805:0.2:3900, x-> res_model1_gk(x, args(res_fit1_4); mrho=mrho_model4),  lab = "Fit 1")
plot!(3805:0.2:3900, x-> res_model1_gk(x, args(res_fit1_4_2); mrho=mrho_model4),  lab = "Fit 2")
plot!(3805:0.2:3900, x-> model1(x, args(res_fit1_4_2); mrho=mrho_model4),  l = (:dash), lab = :none)

xlabel!(L"\sqrt{s}"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow \pi^{+} \pi^{-} {J} / \psi\right)"*" (pb)")

In [ ]:
# Monte Carlo sampling 30000 parameter sets
@time psets2 = sampled_psets(res_fit1_4_2, 30000);
# choose only those with positive normalization as its sign does not matter
psets2 = psets2[findall(psets2[:,1] .>0), :]
@time psets_plt(psets2)

In [ ]:
# choose the parameter sets resulting in χ²≤ χbest²+3.53
δχ2_2 = [chisq((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data, x) - res_fit1_4_2.fval for x in eachrow(psets2) ]
psets2_1σ3p = psets2[findall(<=(3.53), δχ2_2), :]
@time psets_plt(psets2_1σ3p)

In [ ]:
psets2_1σfinal3p = psets2_1σ3p[findall(psets2_1σ3p[:,1] .< 0.3), :]
@time psets_plt(psets2_1σfinal3p)

In [ ]:
pval_errors(res_fit1_4_2, psets2_1σfinal3p)

In [ ]:
ybands2_3p = [yerr(x, res_fit1_4_2, psets2_1σfinal3p) for x = xarr];
ribbon2_low = [ybands2_3p[i][1] for i in eachindex(ybands2_3p)] 
ribbon2_up = [ybands2_3p[i][2] for i in eachindex(ybands2_3p)];

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(xarr, x-> res_model1_gk(x, args(res_fit1_4_2); mrho=mrho_model4), lw = 1.5, lab = "Fit 2"; ribbon = (-ribbon2_low, ribbon2_up))
plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_4_2); mrho=mrho_model4),  l = (:dash), lab = :none)
xlims!(3805, 3900)
xlabel!(raw"$\sqrt{s}$"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow  {J} / \psi \pi^{+} \pi^{-}\right)"*" (pb)")
# savefig("fit2_3p.pdf") 

### Fit 3

In [ ]:
# a different fit, found by a large sampling signaled by some parameter sets far from others
res_fit1_4_3 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data,(0.4, -0.025, -12); 
    name =[:p0, :r, :a22], error=[0.1, 0.01, 1], fix_a22 = false )
migrad(res_fit1_4_3)

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(3805:0.2:3900, x-> res_model1_gk(x, args(res_fit1_4); mrho=mrho_model4),  lab = "Fit 1")
plot!(3805:0.2:3900, x-> res_model1_gk(x, args(res_fit1_4_2); mrho=mrho_model4),  lab = "Fit 2")
plot!(3805:0.2:3900, x-> res_model1_gk(x, args(res_fit1_4_3); mrho=mrho_model4),  lab = "Fit 3")
plot!(3805:0.2:3900, x-> model1(x, args(res_fit1_4_3); mrho=mrho_model4),  l = (:dash), lab = :none)

xlabel!(L"\sqrt{s}"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow \pi^{+} \pi^{-} {J} / \psi\right)"*" (pb)")

In [ ]:
# Monte Carlo sampling 30000 parameter sets
@time psets3 = sampled_psets(res_fit1_4_3, 30000);
# choose only those with positive normalization as its sign does not matter
psets3 = psets3[findall(psets3[:,1] .>0), :]
@time psets_plt(psets3)

In [ ]:
# choose the parameter sets resulting in χ²≤ χbest²+3.53
δχ2_3 = [chisq((x, par) -> res_model1_gk(x, par; mrho=mrho_model4), data, x) - res_fit1_4_3.fval for x in eachrow(psets3) ]
psets3_1σ3p = psets3[findall(<=(3.53), δχ2_3), :]
@time psets_plt(psets3_1σ3p)

In [ ]:
# choose the parameter sets with the first parameter < 0.8, as the others should be from a different fit
psets3_1σfinal3p = psets3_1σ3p[findall(psets3_1σ3p[:,1] .< 0.8), :]
@time psets_plt(psets3_1σfinal3p)

In [ ]:
pval_errors(res_fit1_4_3, psets3_1σfinal3p)

In [ ]:
ybands3_3p = [yerr(x, res_fit1_4_3, psets3_1σfinal3p) for x = xarr];
ribbon3_low = [ybands3_3p[i][1] for i in eachindex(ybands3_3p)] 
ribbon3_up = [ybands3_3p[i][2] for i in eachindex(ybands3_3p)];

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(xarr, x-> res_model1_gk(x, args(res_fit1_4_3); mrho=mrho_model4), lw = 1.5, lab = "Fit 3"; ribbon = (-ribbon3_low, ribbon3_up))
plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_4_3); mrho=mrho_model4),  l = (:dash), lab = :none)
xlims!(3805, 3900)
xlabel!(raw"$\sqrt{s}$"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow  {J} / \psi \pi^{+} \pi^{-}\right)"*" (pb)")
# savefig("fit3_3p.pdf") 

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(xarr, x-> res_model1_gk(x, args(res_fit1_4); mrho=mrho_model4), l = (:solid, 1.5, palette(:default)[1]), lab = "Fit 1"; ribbon = (-ribbon1_low, ribbon1_up), fillalpha=0.2)
plot!(xarr, x-> res_model1_gk(x, args(res_fit1_4_2); mrho=mrho_model4), l = (:dash, 1.5), lab = "Fit 2"; ribbon = (-ribbon2_low, ribbon2_up), fillalpha=0.2)
plot!(xarr, x-> res_model1_gk(x, args(res_fit1_4_3); mrho=mrho_model4), l = (:dot, 1.5), lab = "Fit 3"; ribbon = (-ribbon3_low, ribbon3_up), fillalpha=0.2)
plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_4); mrho=mrho_model4),  l = (:dashdot, palette(:default)[1]), linealpha=0.5, lab = :none)
# plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_4_2); mrho=mrho_model4),  l = (:dash), lab = :none)
# plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_4_3); mrho=mrho_model4),  l = (:dash), lab = :none)
xlims!(3805, 3900)
xlabel!(raw"$\sqrt{s}$"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow  {J} / \psi \pi^{+} \pi^{-}\right)"*" (pb)")
# savefig("allfits_3p.pdf") 

In [ ]:
# add a fake data point to avoid producing a pole at the J/ψρ threshold, which is now 3946.9 MeV
fakesets1 = Data([3847], [16.0], [3]);
fakedata1 = vcat(data, fakesets1)

In [ ]:
mrho_model= 2mπc;
res_fit1_1 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model), data,(3.4, 0.0022, -7.5); name =[:p0, "p1r/p0", :a22], error=[0.1, 0.001, 0.1], fix_a22 = false)
migrad(res_fit1_1)
hesse(res_fit1_1)
hesse(res_fit1_1)
migrad(res_fit1_1)
minos(res_fit1_1)

In [ ]:
@plt_data data lab = "BESIII data"
plot!(3805:0.1:3900, x-> res_model1_gk(x, args(res_fit1_1); mrho=mrho_model), lab = "Fit w/ "*L"a_{22,{\rm eff}}=(-6.39+i11.74)"*" fm" )
xlabel!(L"\sqrt{s}"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow \pi^{+} \pi^{-} {J} / \psi\right)"*" (pb)")
# savefig("fit1.pdf")
# title!("this fit produces a pole around "*L"$J/\psi\rho$"*" threhold; "*L"m_\rho="*"$mrho_model"*" MeV")

In [ ]:
mrho_model2 = 600
res_fit1_2 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model2), data,(4, 0.0018, -5); name =[:p0, "p1r/p0", :a22], error=[0.1, 0.001, 0.1], fix_a22 = false)
migrad(res_fit1_2)
hesse(res_fit1_2)
hesse(res_fit1_2)
migrad(res_fit1_2)
minos(res_fit1_2)

In [ ]:
@plt_data data lab = "BESIII data"
plot!(3805:0.1:3900, x-> res_model1_gk(x, args(res_fit1_1); mrho=mrho_model), lab = "Fit w/ "*L"M_\rho=2M_\pi" )
plot!(3805:0.1:3900, x-> res_model1_gk(x, args(res_fit1_2); mrho=mrho_model2), l=:dash,  lab = "Fit w/ "*L"M_\rho=0.6"*" GeV")
plot!(3805:0.1:3900, x-> model1(x, args(res_fit1_1); mrho=mrho_model), l= (palette(:default)[2], :dot), lab=:none )
# plot!(3805:0.1:3900, x-> res_model1_gk(x, args(res_fit1_3); mrho=mrho_model3), l=:dash,  lab = "Fit w/ "*L"M_\rho=0.6"*" GeV")
xlabel!(L"\sqrt{s}"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow \pi^{+} \pi^{-} {J} / \psi\right)"*" (pb)")
# savefig("fit1.pdf")

In [ ]:
mrho_model3 = 650
res_fit1_3 = model_fit((x, par) -> res_model1_gk(x, par; mrho=mrho_model3), data,(3.4, 0.001, -7.5); name =[:p0, "p1r/p0", :a22], error=[0.1, 0.0001, 0.1], fix_a22 = false)
migrad(res_fit1_3)
minos(res_fit1_3)

# with a non-interfering background

In [ ]:
# here we define r = p1r/p0, p0 is an overall normalization constant
# introduce a non-interfering bg   Nov.08, 2024
function model1_bg(x, par; mrho = mρ)
    p0, bg, a22 = par
    a22 = a22/ħc
    a22eff = (-6.39 + 11.74im) /ħc; inv_a22eff = 1/a22eff
    k1 = real( xsqrt(qsq(x, mjψ, mrho)) ) 
    a11 = 1/( k1/imag(inv_a22eff) * (real(inv_a22eff) - 1/a22) ) 
    amp2 = bg^2 + p0^2 * (abs2( t11_lo_constained(x, a11, a22, a22eff; mrho)) )
    return amp2
end

XW[] = gauss(128, 10, -10)
function res_model1_bg(x, σ, par; mrho = mρ)
    _y, _w = XW[]
    return quadgauss(y -> model1_bg(x-y, par; mrho) * resolution(y, σ), _y, _w)
end

res_model1_bg(x, par; mrho = mρ) = res_model1_bg(x, 1.7, par; mrho)

In [ ]:
# to use an adaptive integration method to have better precision over possible narrow pole
function res_model1_gk_bg(x, σ, par; mrho=mρ)
    return quadgk(y -> model1_bg(x-y, par; mrho) * resolution(y, σ), -10, 10, rtol=1e-3)[1]
end
res_model1_gk_bg(x, par; mrho=mρ) = res_model1_gk_bg(x, 1.7, par; mrho)

In [ ]:
res_model1_gk_bg(3820, (0.,4,-2.2)), res_model1_gk_bg(3865, (0.1,4,-1.2))

In [ ]:
# this is the correct choice.
mrho_model4 = 775 - 75im
res_fit1_4_bg = model_fit((x, par) -> res_model1_gk_bg(x, par; mrho=mrho_model4), data,(0.1, 3.2, -5); 
    name =[:p0, :bg, :a22], error=[0.1, 0.01, 1], fix_a22 = false, fix_bg =true )
migrad(res_fit1_4_bg)
# simplex(res_fit1_4_bg)
migrad(res_fit1_4_bg)
migrad(res_fit1_4_bg)
# minos(res_fit1_4_bg)

In [ ]:
@plt_data data lab = "BESIII data" c=:black l = :black
plot!(3805:0.2:3900, x-> res_model1_gk_bg(x, args(res_fit1_4_bg); mrho=mrho_model4),  lab = "Best fit")
plot!(3805:0.2:3900, x-> model1_bg(x, args(res_fit1_4_bg); mrho=mrho_model4),  l = (:dash), lab = :none)

xlabel!(L"\sqrt{s}"*" (MeV)")
ylabel!(L"\sigma\left({e}^{+} {e}^{-} \rightarrow \pi^{+} \pi^{-} {J} / \psi\right)"*" (pb)")

In [ ]:
plot(3805:0.2:3900, x-> res_model1_gk_bg(x, (0.01, 0, -11.3); mrho=mrho_model4),   lab = :none)
plot!(3805:0.2:3900, x-> model1_bg(x, (0.01, 1, -11.3); mrho=mrho_model4),   lab = :none)
